# PH Address Normalization & Fuzzy Matching Pipeline

**Part 1:** Normalize raw addresses → detect city (right-to-left) → filter `dim_location`  
**Part 2:** Fuzzy match to filtered `dim_location` → export `valid` / `invalid` Excel files

---

## 0. Install Dependencies

## 1. Imports & Configuration

Update the **file paths** and **`RAW_ADDRESSES`** list below to point at your actual data.

In [104]:
import os
import re
import unicodedata
import pandas as pd
from rapidfuzz import fuzz, process

# ── File Paths (update as needed) ─────────────────────────────────────────────
DIM_LOC_PATH = "../../data/mapping/dim_location_20260415_v3.csv"
ALIAS_PATH   = "../../data/utils/ph_address_alias_extended_v4.csv"
VALID_DIR    = "../../data/output/valid"
INVALID_DIR  = "../../data/output/invalid"

os.makedirs(VALID_DIR,   exist_ok=True)
os.makedirs(INVALID_DIR, exist_ok=True)

# ── Fuzzy match threshold (0–100) ─────────────────────────────────────────────
FUZZY_THRESHOLD = 70

print("Config OK ✓")

Config OK ✓


## 2. Raw Address Input

Replace the list below with your actual addresses, **or** load from a CSV/Excel file using the commented-out alternative.

In [105]:

# Load from file ──────────────────────────────────────────────────
psaddress_df    = pd.read_excel("../../data/sample/ps_address_v3.xlsx")
# psaddress_df = psaddress_df.head(1000)
RAW_ADDRESSES = psaddress_df["order_deliveraddress"].dropna().tolist()

print(f"Loaded {len(RAW_ADDRESSES)} raw addresses")

Loaded 122888 raw addresses


## 3. Load Reference Data

Loads `dim_location` (42k+ barangay records) and `ph_address_alias` (515 normalization rules).

In [106]:
print("Loading reference data …")

# dim_location
dim_raw = pd.read_csv(DIM_LOC_PATH, encoding="latin1")
dim_raw.columns = dim_raw.columns.str.strip()
for col in ["barangay_name", "city_name", "province_name", "region_name"]:
    dim_raw[col] = dim_raw[col].astype(str).str.strip()

# alias rules
alias_df = pd.read_csv(ALIAS_PATH, encoding="latin1", usecols=["pattern", "replacement"])
alias_df = alias_df.dropna(subset=["pattern", "replacement"])
alias_df["pattern"]     = alias_df["pattern"].astype(str).str.strip()
alias_df["replacement"] = alias_df["replacement"].astype(str).str.strip()
# Sort longest-first so longer patterns match before shorter ones
alias_df = alias_df.sort_values("pattern", key=lambda s: s.str.len(), ascending=False)
# Build lookup: UPPER pattern → UPPER replacement
alias_map = dict(zip(alias_df["pattern"].str.upper(), alias_df["replacement"].str.upper()))

print(f"  dim_location rows : {len(dim_raw):,}")
print(f"  alias rules       : {len(alias_map):,}")
dim_raw.head(3)

Loading reference data …
  dim_location rows : 42,011
  alias rules       : 520


,psgc_10_digit,region_name,province_name,city_name,barangay_name,region_code,province_code,city_code,barangay_code
0,1400101001,Cordillera Administrative Region (CAR),Abra,Bangued,Agtangao,14,1,1,1
1,1400101002,Cordillera Administrative Region (CAR),Abra,Bangued,Angad,14,1,1,2
2,1400101003,Cordillera Administrative Region (CAR),Abra,Bangued,Bañacao,14,1,1,3


**Reasoning**:
The subtask requires adding cleaned versions of 'city_name', 'province_name', and 'barangay_name' to the `dim_raw` DataFrame. I will apply the `clean_str` function to these columns and then display the head of the DataFrame to verify the new columns. This new cell will be inserted after `cell-load-ref` where `dim_raw` is loaded.



In [107]:
print("Adding cleaned columns to dim_raw...")
dim_raw["city_clean"] = dim_raw["city_name"].apply
dim_raw["prov_clean"] = dim_raw["province_name"].apply
dim_raw["bgy_clean"]  = dim_raw["barangay_name"].apply

print("Cleaned columns added to dim_raw ✓")
dim_raw.head()

Adding cleaned columns to dim_raw...
Cleaned columns added to dim_raw ✓


,psgc_10_digit,region_name,province_name,city_name,barangay_name,region_code,province_code,city_code,barangay_code,city_clean,prov_clean,bgy_clean
0,1400101001,Cordillera Administrative Region (CAR),Abra,Bangued,Agtangao,14,1,1,1,<bound method Series.apply of 0 Bangue...,<bound method Series.apply of 0 ...,<bound method Series.apply of 0 Agtan...
1,1400101002,Cordillera Administrative Region (CAR),Abra,Bangued,Angad,14,1,1,2,<bound method Series.apply of 0 Bangue...,<bound method Series.apply of 0 ...,<bound method Series.apply of 0 Agtan...
2,1400101003,Cordillera Administrative Region (CAR),Abra,Bangued,Bañacao,14,1,1,3,<bound method Series.apply of 0 Bangue...,<bound method Series.apply of 0 ...,<bound method Series.apply of 0 Agtan...
3,1400101004,Cordillera Administrative Region (CAR),Abra,Bangued,Bangbangar,14,1,1,4,<bound method Series.apply of 0 Bangue...,<bound method Series.apply of 0 ...,<bound method Series.apply of 0 Agtan...
4,1400101005,Cordillera Administrative Region (CAR),Abra,Bangued,Cabuloan,14,1,1,5,<bound method Series.apply of 0 Bangue...,<bound method Series.apply of 0 ...,<bound method Series.apply of 0 Agtan...


## 4. Helper Functions

In [108]:
def strip_accents(text: str) -> str:
    """Remove diacritical marks (e.g. ñ → n, é → e)."""
    return "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )

def clean_str(s: str) -> str:
    """Lowercase, strip accents, collapse whitespace."""
    s = strip_accents(str(s)).lower()
    return re.sub(r"\s+", " ", s).strip()

print("Helper functions defined ✓")

Helper functions defined ✓


**Reasoning**:
The subtask requires loading the `dictionary.json` file. I will import the `json` module, load the file into a variable, and then print the keys and a sample of its content to understand its structure.



In [109]:
import json

print("Loading blocking dictionary...")
BLOCKING_DICTIONARY_PATH = "../../data/dictionary/dictionary.json"

with open(BLOCKING_DICTIONARY_PATH, "r") as f:
    blocking_dictionary = json.load(f)

print(f"Blocking dictionary loaded with {len(blocking_dictionary):,} entries.")
print("First 5 keys:")
for i, key in enumerate(blocking_dictionary.keys()):
    if i >= 5:
        break
    print(f"- {key}")

print("\nSample entry for 'city_province_map':")
if "city_province_map" in blocking_dictionary:
    sample_map = {k: blocking_dictionary["city_province_map"][k] for k in list(blocking_dictionary["city_province_map"])[:3]}
    print(sample_map)
else:
    print("Key 'city_province_map' not found.")

Loading blocking dictionary...
Blocking dictionary loaded with 18 entries.
First 5 keys:
- Bangsamoro Autonomous Region In Muslim Mindanao (BARMM)
- Cordillera Administrative Region (CAR)
- MIMAROPA Region
- National Capital Region (NCR)
- Negros Island Region (NIR)

Sample entry for 'city_province_map':
Key 'city_province_map' not found.


**Reasoning**:
Before defining the `detect_city_improved` function, it's essential to prepare the necessary lookup dictionaries (`city_to_provinces`, `province_to_regions`, `all_cities_clean`) from the loaded `dim_raw` and `blocking_dictionary`. This ensures that all required data structures are available and correctly populated for the enhanced city detection logic. This step extracts the hierarchical relationships (region-province-city) from the `blocking_dictionary` and combines them with `dim_raw` data for comprehensive lookups.



In [110]:
# Helper to build city_to_provinces and province_to_regions
city_to_provinces = {}
province_to_regions = {}

for region_name, provinces_data in blocking_dictionary.items():
    clean_region = clean_str(region_name)
    for province_name, cities_data in provinces_data.items():
        clean_province = clean_str(province_name)
        province_to_regions.setdefault(clean_province, set()).add(clean_region)

        for city_name, city_aliases_data in cities_data["cities"].items():
            clean_city = clean_str(city_name)
            city_to_provinces.setdefault(clean_city, set()).add(clean_province)
            # Also add aliases as city keys pointing to the same province set
            for alias in city_aliases_data.get("aliases", []):
                clean_alias = clean_str(alias)
                city_to_provinces.setdefault(clean_alias, set()).add(clean_province)

# Recreate all_cities and all_cities_clean to ensure it's comprehensive and available
all_cities = dim_raw["city_name"].dropna().unique()
all_cities_clean = {clean_str(c): c for c in all_cities}   # clean → original
all_city_cleans = list(all_cities_clean.keys())

print("Blocking dictionary processed into city_to_provinces, province_to_regions, and all_cities_clean ✓")

Blocking dictionary processed into city_to_provinces, province_to_regions, and all_cities_clean ✓


---
## PART 1 — Normalize Addresses

Applies alias replacements token-by-token (up to 3-word patterns, longest-first).  
Examples: `BRGY.` → `BARANGAY`, `STA` → `SANTA`, `AVE` → `AVENUE`, `GENSAN` → `GENERAL SANTOS`

In [111]:
import re

# Preprocess alias_map into tuple keys (DO THIS ONCE)
alias_map_tokens = {
    tuple(k.split()): v for k, v in alias_map.items()
}

max_alias_len = max(len(k) for k in alias_map_tokens)


def normalize_address(addr: str) -> str:
    if pd.isna(addr):
        return ""

    upper = str(addr).upper()

    # Faster tokenization (no capturing groups)
    tokens = upper.replace(",", " , ").replace(".", " . ").split()

    result = []
    i = 0
    n = len(tokens)

    while i < n:
        if tokens[i] in {",", "."}:
            result.append(tokens[i])
            i += 1
            continue

        matched = False

        # Try longest match first
        for l in range(min(max_alias_len, n - i), 0, -1):
            candidate = tuple(tokens[i:i+l])

            if candidate in alias_map_tokens:
                result.append(alias_map_tokens[candidate])
                i += l
                matched = True
                break

        if not matched:
            result.append(tokens[i])
            i += 1

    return " ".join(result).replace(" ,", ",").replace(" .", ".").title()

In [112]:
print("Normalizing addresses …")
records = [
    {"order_deliveraddress": addr, "normalized_address": normalize_address(addr)}
    for addr in RAW_ADDRESSES
]
normalized_df = pd.DataFrame(records)

print(f"Done — {len(normalized_df)} addresses normalized")
normalized_df

Normalizing addresses …
Done — 122888 addresses normalized


,order_deliveraddress,normalized_address
0,Santo Domingo St,Santo Domingo Street
1,san bartolome,San Bartolome
2,9036 Manggahan,9036 Manggahan
3,Tagumpay sindalan CSFP,Tagumpay Sindalan San Fernando
4,"BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL...","Block 16, Lot 4 Quiotan Street, Phil Am Life V..."
...,...,...
122883,VILLA SANTI,Villa Santi
122884,PARIANCILLO,Pariancillo
122885,Fiesta Comm. Block 16 lot 38-40,Fiesta Comm. Block 16 Lot 38-40
122886,1924 TRINIDAD ST TONDO MANILA,1924 Trinidad Street Tondo Manila


### 1b. City Detection (Right-to-Left)

Reads each address from **right to left** (comma-delimited segments).  
Priority: exact word-boundary match → fuzzy `token_set_ratio ≥ 85`.

In [113]:
from functools import lru_cache

@lru_cache(maxsize=200_000)
def clean_cached(value: str) -> str:
    return clean_str(value)

def _segments_rtl(norm_addr: str) -> list[str]:
    if pd.isna(norm_addr):
        return []
    segments = [segment.strip() for segment in str(norm_addr).split(",")]
    segments = [segment for segment in segments if segment]
    return list(reversed(segments)) or [str(norm_addr)]

def _phrases(tokens: list[str], max_words: int = 4):
    for size in range(min(max_words, len(tokens)), 0, -1):
        for start in range(len(tokens) - size + 1):
            yield " ".join(tokens[start:start + size])

# Build city/province lookup from dim_location
all_cities = dim_raw["city_name"].dropna().astype(str).unique().tolist()
all_provinces = dim_raw["province_name"].dropna().astype(str).unique().tolist()

all_cities_clean = {clean_cached(city): city for city in all_cities}
all_provinces_clean = {clean_cached(prov): prov for prov in all_provinces}
all_city_cleans = tuple(all_cities_clean.keys())
all_province_cleans = tuple(all_provinces_clean.keys())

city_to_provinces = {}
province_to_regions = {}
city_alias_to_clean = {}

for region_name, provinces_data in blocking_dictionary.items():
    clean_region = clean_cached(region_name)
    for province_name, cities_data in provinces_data.items():
        clean_province = clean_cached(province_name)
        province_to_regions.setdefault(clean_province, set()).add(clean_region)

        for city_name, city_aliases_data in cities_data["cities"].items():
            clean_city = clean_cached(city_name)
            city_to_provinces.setdefault(clean_city, set()).add(clean_province)
            city_alias_to_clean[clean_city] = clean_city

            for alias in city_aliases_data.get("aliases", []):
                clean_alias = clean_cached(alias)
                city_to_provinces.setdefault(clean_alias, set()).add(clean_province)
                city_alias_to_clean[clean_alias] = clean_city

province_clean_to_original = {clean_cached(p): p for p in all_provinces}

# Manila in addresses is often generic; in dim_location these are district-level city names.
MANILA_CANONICAL_CITY = "__MANILA__"
MANILA_DISTRICTS = {
    "binondo", "ermita", "intramuros", "malate", "paco", "pandacan",
    "port area", "quiapo", "sampaloc", "san miguel", "san nicolas",
    "santa ana", "santa cruz", "tondo i/ii",
}

manila_rows = dim_raw[
    (dim_raw["province_name"].apply(clean_cached) == "metro manila")
    & (dim_raw["city_name"].apply(clean_cached).isin(MANILA_DISTRICTS))
].copy()
manila_city_clean_to_orig = {
    clean_cached(c): c for c in manila_rows["city_name"].dropna().unique()
}
manila_city_cleans = list(manila_city_clean_to_orig.keys())

def detect_city_candidates(norm_addr: str) -> list[tuple[str, str]]:
    candidates = []
    seen = set()

    for seg in _segments_rtl(norm_addr):
        seg_c = clean_cached(seg)
        if len(seg_c) < 3:
            continue

        tokens = seg_c.split()

        for phrase in _phrases(tokens, max_words=4):
            city_key = city_alias_to_clean.get(phrase)
            if city_key is None and phrase in all_cities_clean:
                city_key = phrase
            if city_key is None:
                continue

            city_orig = all_cities_clean.get(city_key, city_key)
            for prov_clean in city_to_provinces.get(city_key, set()):
                prov_orig = province_clean_to_original.get(prov_clean)
                if prov_orig:
                    pair = (city_orig, prov_orig)
                    if pair not in seen:
                        seen.add(pair)
                        candidates.append(pair)
            if candidates:
                return candidates

        best = process.extractOne(
            seg_c,
            all_city_cleans,
            scorer=fuzz.token_set_ratio,
            score_cutoff=85,
        )
        if best:
            city_key = best[0]
            city_orig = all_cities_clean[city_key]
            for prov_clean in city_to_provinces.get(city_key, set()):
                prov_orig = province_clean_to_original.get(prov_clean)
                if prov_orig:
                    pair = (city_orig, prov_orig)
                    if pair not in seen:
                        seen.add(pair)
                        candidates.append(pair)

    return candidates

def detect_province(norm_addr: str):
    for seg in _segments_rtl(norm_addr):
        seg_c = clean_cached(seg)
        if len(seg_c) < 3:
            continue

        tokens = seg_c.split()
        for phrase in _phrases(tokens, max_words=4):
            prov_orig = all_provinces_clean.get(phrase)
            if prov_orig:
                return prov_orig

        best = process.extractOne(
            seg_c,
            all_province_cleans,
            scorer=fuzz.token_set_ratio,
            score_cutoff=80,
        )
        if best:
            return all_provinces_clean[best[0]]

    if re.search(r"\bmanila\b", clean_cached(norm_addr)):
        return all_provinces_clean.get("metro manila")

    return None

print("detect_city_candidates() and detect_province() defined ✓")

detect_city_candidates() and detect_province() defined ✓


**Reasoning**:
To fulfill the requirement of returning original casing for province names in the `detect_city_improved` function, I need to create a mapping from cleaned province names to their original forms. This mapping will be generated from the `province_name` column of the `dim_raw` DataFrame, ensuring that the canonical province names retain their original casing.



In [114]:
print("Preparing province clean to original mapping...")
province_clean_to_original = {clean_str(p): p for p in dim_raw["province_name"].dropna().unique()}
print("Province clean to original mapping ready ✓")

def detect_city_improved(norm_addr: str) -> list[tuple[str, str]]:
    """
    Right-to-left city detection across comma-delimited segments.
    Returns a list of (canonical_city_name, canonical_province_name) tuples if found,
    or an empty list if no city is detected.
    """
    candidate_pairs = set()  # Use set to store unique (city, province) candidates
    segments_rtl = list(reversed([s.strip() for s in norm_addr.split(" ")]))

    for seg in segments_rtl:
        seg_c = clean_str(seg)
        if len(seg_c) < 3: # Skip very short segments for efficiency
            continue

        # 1) Exact word-boundary match (longest city wins)
        matched_city_found_in_segment = False
        for city_c, city_orig in sorted(all_cities_clean.items(), key=lambda x: -len(x[0])):
            if re.search(r"\b" + re.escape(city_c) + r"\b", seg_c):
                provinces_for_city = city_to_provinces.get(city_c, set())
                for prov_clean in provinces_for_city:
                    original_prov = province_clean_to_original.get(prov_clean)
                    if original_prov:
                        candidate_pairs.add((city_orig, original_prov))
                matched_city_found_in_segment = True
                break # Break after first exact match in segment (longest one)

        if matched_city_found_in_segment:
            continue # Move to next segment if exact match found

        # 2) Fuzzy fallback per segment if no exact match found
        best = process.extractOne(seg_c, all_city_cleans, scorer=fuzz.token_set_ratio)
        if best and best[1] >= 85:
            matched_city_clean = best[0]
            matched_city_original = all_cities_clean[matched_city_clean]
            provinces_for_city = city_to_provinces.get(matched_city_clean, set())
            for prov_clean in provinces_for_city:
                original_prov = province_clean_to_original.get(prov_clean)
                if original_prov:
                    candidate_pairs.add((matched_city_original, original_prov))

    return list(candidate_pairs)

print("detect_city_improved() defined ✓")

Preparing province clean to original mapping...
Province clean to original mapping ready ✓
detect_city_improved() defined ✓


In [115]:
print("Detecting cities/provinces …")
normalized_df["city_candidates"] = normalized_df["normalized_address"].apply(detect_city_candidates)
normalized_df["detected_city"] = normalized_df["city_candidates"].apply(lambda x: x[0][0] if x else None)
normalized_df["detected_province"] = normalized_df["city_candidates"].apply(lambda x: x[0][1] if x else None)

missing_prov_mask = normalized_df["detected_province"].isna()
normalized_df.loc[missing_prov_mask, "detected_province"] = normalized_df.loc[missing_prov_mask, "normalized_address"].apply(detect_province)

has_city = normalized_df[normalized_df["detected_city"].notna()].copy()
no_city = normalized_df[normalized_df["detected_city"].isna()].copy()

print(f"  ✅ Addresses with city       : {len(has_city)}")
print(f"  ❌ Addresses without city    : {len(no_city)} -> invalid")
print(f"  ℹ️ Addresses with province   : {has_city['detected_province'].notna().sum()}")
normalized_df[["order_deliveraddress", "normalized_address", "detected_city", "detected_province", "city_candidates"]]

Detecting cities/provinces …


KeyboardInterrupt: 

### 1c. Filter dim_location to Detected Cities

Reduces the 42k-row reference table to only the cities present in the address batch — speeds up fuzzy matching significantly.

In [ ]:
detected_cities = has_city["detected_city"].dropna().unique()
detected_provinces = has_city["detected_province"].dropna().unique()

# Keep filtering simple: include detected cities OR detected provinces.
city_mask = dim_raw["city_name"].isin(detected_cities)
prov_mask = dim_raw["province_name"].isin(detected_provinces) if len(detected_provinces) else False
dim_filtered = dim_raw[city_mask | prov_mask].copy()

# Pre-clean columns for matching
dim_filtered["city_clean"] = dim_filtered["city_name"].apply(clean_str)
dim_filtered["prov_clean"] = dim_filtered["province_name"].apply(clean_str)
dim_filtered["bgy_clean"] = dim_filtered["barangay_name"].apply(clean_str)

print(f"Filtered dim_location: {len(dim_raw):,} -> {len(dim_filtered):,} rows")
print(f"Cities included: {list(detected_cities)}")
print(f"Provinces included: {list(detected_provinces)}")

dim_filtered.head(10)

Filtered dim_location: 42,011 -> 0 rows
Cities included: []
Provinces included: []


,psgc_10_digit,region_name,province_name,city_name,barangay_name,region_code,province_code,city_code,barangay_code,city_clean,prov_clean,bgy_clean


---
## PART 2 — Fuzzy Matching to dim_location

For each address:
1. Sub-filters `dim_location` to the detected city
2. Fuzzy-matches the full address string against barangay names (`partial_ratio ≥ FUZZY_THRESHOLD`)
3. Computes `city_match`, `province_match`, `city_match_fuzzy`, `province_match_fuzzy` flags

In [ ]:
def fuzzy_match_flag(a: str, b: str, threshold: int = FUZZY_THRESHOLD) -> bool:
    return fuzz.partial_ratio(str(a), str(b)) >= threshold

def match_address(row) -> dict:
    norm = row["normalized_address"]
    addr_c = clean_cached(norm)
    det_city = row.get("detected_city")
    det_prov = row.get("detected_province")
    candidates = row.get("city_candidates") or []

    if candidates:
        scope_candidates = candidates
    elif pd.notna(det_city):
        scope_candidates = [(det_city, det_prov if pd.notna(det_prov) else None)]
    else:
        scope_candidates = []

    best_row = None
    best_score = -1

    for cand_city, cand_prov in scope_candidates:
        if cand_city == MANILA_CANONICAL_CITY:
            sub = manila_rows.copy()
            city_terms_for_addr = {"manila"}
        else:
            sub = dim_filtered[dim_filtered["city_name"] == cand_city].copy()
            city_terms_for_addr = {clean_cached(cand_city)} if pd.notna(cand_city) else set()

        if pd.notna(cand_prov):
            narrowed = sub[sub["province_name"] == cand_prov]
            if not narrowed.empty:
                sub = narrowed

        if sub.empty:
            continue

        bgy_names = sub["bgy_clean"].tolist()
        best_match = process.extractOne(
            addr_c,
            bgy_names,
            scorer=fuzz.partial_ratio,
            score_cutoff=FUZZY_THRESHOLD,
        )

        if best_match:
            matched_row = sub[sub["bgy_clean"] == best_match[0]].iloc[0]
            if best_match[1] > best_score:
                best_row = matched_row
                best_score = best_match[1]
        elif best_row is None:
            best_row = sub.iloc[0]

    if best_row is None:
        return {
            "barangay_code": None, "barangay_name": None,
            "city_code": None, "city_name": det_city,
            "province_code": None, "province_name": det_prov if pd.notna(det_prov) else None,
            "region_code": None, "region_name": None,
            "addr_clean": addr_c,
            "city_clean": clean_cached(det_city) if pd.notna(det_city) else "",
            "prov_clean": clean_cached(det_prov) if pd.notna(det_prov) else "",
            "city_match": False, "province_match": False,
            "city_match_fuzzy": False, "province_match_fuzzy": False,
            "match_status": "city_only",
        }

    city_c = clean_cached(best_row["city_name"]).strip()
    prov_c = clean_cached(best_row["province_name"]).strip()

    city_terms = {city_c}
    if pd.notna(det_city):
        city_terms.add(clean_cached(det_city))

    province_terms = {prov_c}
    if pd.notna(det_prov):
        province_terms.add(clean_cached(det_prov))
    if prov_c == "metro manila":
        province_terms.add("manila")

    city_match = any(term and term in addr_c for term in city_terms)
    province_match = any(term and term in addr_c for term in province_terms)
    city_match_fuzzy = any(fuzzy_match_flag(term, addr_c) for term in city_terms if term)
    province_match_fuzzy = any(fuzzy_match_flag(term, addr_c) for term in province_terms if term)

    if pd.notna(best_row["barangay_code"]):
        match_status = "valid"
    elif (city_match or city_match_fuzzy) and (province_match or province_match_fuzzy):
        match_status = "valid_no_barangay"
    else:
        match_status = "city_only"

    return {
        "barangay_code": best_row["barangay_code"],
        "barangay_name": best_row["barangay_name"],
        "city_code": best_row["city_code"],
        "city_name": best_row["city_name"],
        "province_code": best_row["province_code"],
        "province_name": best_row["province_name"],
        "region_code": best_row["region_code"],
        "region_name": best_row["region_name"],
        "addr_clean": addr_c,
        "city_clean": city_c,
        "prov_clean": prov_c,
        "city_match": city_match,
        "province_match": province_match,
        "city_match_fuzzy": city_match_fuzzy,
        "province_match_fuzzy": province_match_fuzzy,
        "match_status": match_status,
    }

print("match_address() defined ✓")

match_address() defined ✓


In [ ]:
print("Running fuzzy matching …")
matched_results = has_city.apply(match_address, axis=1, result_type="expand")
valid_df = pd.concat(
    [has_city.reset_index(drop=True), matched_results.reset_index(drop=True)],
    axis=1
)

print(f"Matching complete — {len(valid_df)} addresses processed")
valid_df[["order_deliveraddress", "city_name", "province_name", "barangay_name",
          "city_match", "city_match_fuzzy", "province_match_fuzzy", "match_status"]]

Running fuzzy matching …
Matching complete — 0 addresses processed


KeyError: "['city_name', 'province_name', 'barangay_name', 'city_match', 'city_match_fuzzy', 'province_match_fuzzy', 'match_status'] not in index"

## 5. Segregate Valid / Invalid

In [ ]:
OUTPUT_COLS = [
    "order_deliveraddress", "normalized_address",
    "barangay_code", "barangay_name",
    "city_code", "city_name",
    "province_code", "province_name",
    "region_code", "region_name",
    "addr_clean", "city_clean", "prov_clean",
    "city_match", "province_match",
    "city_match_fuzzy", "province_match_fuzzy",
    "match_status",
]

def ensure_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = None
    return df[cols]

VALID_STATUSES = {"valid", "valid_no_barangay"}

valid_final = valid_df[valid_df["match_status"].isin(VALID_STATUSES)].copy()
invalid_part2 = valid_df[~valid_df["match_status"].isin(VALID_STATUSES)].copy()
invalid_final = pd.concat(
    [no_city.assign(match_status="no_city_detected"), invalid_part2],
    ignore_index=True
)

valid_out = ensure_cols(valid_final.copy(), OUTPUT_COLS)
invalid_out = ensure_cols(invalid_final.copy(), OUTPUT_COLS)

print(f"✅ Valid   : {len(valid_out)} addresses")
print(f"❌ Invalid : {len(invalid_out)} addresses")

✅ Valid   : 34452 addresses
❌ Invalid : 88436 addresses


### Preview Results

In [ ]:
print("=== VALID ===")
valid_out

=== VALID ===


,order_deliveraddress,normalized_address,barangay_code,barangay_name,city_code,city_name,province_code,province_name,region_code,region_name,addr_clean,city_clean,prov_clean,city_match,province_match,city_match_fuzzy,province_match_fuzzy,match_status
0,Santo Domingo St,Santo Domingo Street,9.0,Santo Domingo Pob.,16.0,Santo Domingo,5.0,Albay,5.0,Region V (Bicol Region),santo domingo street,santo domingo,albay,True,False,True,False,valid
2,"BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL...","Block 16, Lot 4 Quiotan Street, Phil Am Life V...",3.0,Carmen,6.0,Carmen,68.0,Surigao del Sur,16.0,Region XIII (Caraga),"block 16, lot 4 quiotan street, phil am life v...",carmen,surigao del sur,True,False,True,False,valid
3,Cainta Rizal,Cainta Rizal,18.0,Santa Rosa,5.0,Cainta,58.0,Rizal,4.0,Region IV-A (CALABARZON),cainta rizal,cainta,rizal,True,True,True,True,valid
4,2079 a elias st sta cruz,2079 A Elias Street Santa Cruz,23.0,Santisima Cruz,26.0,Santa Cruz,34.0,Laguna,4.0,Region IV-A (CALABARZON),2079 a elias street santa cruz,santa cruz,laguna,True,False,True,False,valid
6,tapaz capiz,Tapaz Capiz,1.0,Abangay,17.0,Tapaz,19.0,Capiz,6.0,Region VI (Western Visayas),tapaz capiz,tapaz,capiz,True,True,True,True,valid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44389,2288 ARAGON ST SAN ANDRES STA ANA MANILA,2288 Aragon Street San Andres Santa Ana Manila,5.0,Tala,40.0,San Andres,56.0,Quezon,4.0,Region IV-A (CALABARZON),2288 aragon street san andres santa ana manila,san andres,quezon,True,False,True,False,valid
44394,Brgy. IV Catanauan Quezon,Barangay. Iv Catanauan Quezon City,60.0,Mariana,4.0,Quezon City,74.0,Metro Manila,13.0,National Capital Region (NCR),barangay. iv catanauan quezon city,quezon city,metro manila,True,False,True,False,valid
44395,328 P.SEVILLA ST,328 P. Sevilla Street,1.0,Bayawahan,39.0,Sevilla,12.0,Bohol,7.0,Region VII (Central Visayas),328 p. sevilla street,sevilla,bohol,True,False,True,False,valid
44396,Zone 2 San Juan Molo SALEM FOUNDATION,Zone 2 San Juan City Molo Salem Foundation,24.0,Muzon,23.0,San Juan,10.0,Batangas,4.0,Region IV-A (CALABARZON),zone 2 san juan city molo salem foundation,san juan,batangas,True,False,True,False,valid


In [ ]:
print("=== INVALID ===")
invalid_out

=== INVALID ===


,order_deliveraddress,normalized_address,barangay_code,barangay_name,city_code,city_name,province_code,province_name,region_code,region_name,addr_clean,city_clean,prov_clean,city_match,province_match,city_match_fuzzy,province_match_fuzzy,match_status
0,san bartolome,San Bartolome,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_city_detected
1,9036 Manggahan,9036 Manggahan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_city_detected
2,ZURI HOTEL,Zuri Hotel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_city_detected
3,9F ST ANDREW ST JPA SUBD,9F Street Andrew Street Jpa Subdivision,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_city_detected
4,blk 15 lot 41 resettlement,Block 15 Lot 41 Resettlement,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_city_detected
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88431,BLOCK 10 LOT 13 PHASE 2 CITATION RESIDENCES BR...,Block 10 Lot 13 Phase 2 Citation Residences Ba...,NaN,NaN,NaN,city of sto. tomas,NaN,Batangas,NaN,NaN,block 10 lot 13 phase 2 citation residences ba...,city of sto. tomas,batangas,False,False,False,False,city_only
88432,"Bloomingdale, Phase 3, Block 33, Lot 4, Iponan...","Bloomingdale, Phase 3, Block 33, Lot 4, Iponan...",NaN,NaN,NaN,city of cagayan de oro,NaN,Misamis Oriental,NaN,NaN,"bloomingdale, phase 3, block 33, lot 4, iponan...",city of cagayan de oro,misamis oriental,False,False,False,False,city_only
88433,400 STA ROSA DE LIMA RD,400 Santa Rosa De Lima Road,NaN,NaN,NaN,city of santa rosa,NaN,Laguna,NaN,NaN,400 santa rosa de lima road,city of santa rosa,laguna,False,False,False,False,city_only
88434,"CANITOAN, CAGAYAN DE ORO","Canitoan, Cagayan De Oro",NaN,NaN,NaN,city of cagayan de oro,NaN,Misamis Oriental,NaN,NaN,"canitoan, cagayan de oro",city of cagayan de oro,misamis oriental,False,False,False,False,city_only


## 6. Export to Excel

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment

def write_excel(df: pd.DataFrame, path: str, sheet: str = "Results"):
    """Write DataFrame to a styled Excel file with frozen header row."""
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name=sheet)
        ws = writer.sheets[sheet]
        # Auto-fit column widths
        for col_cells in ws.columns:
            max_len = max((len(str(c.value)) if c.value else 0) for c in col_cells)
            ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 4, 60)
        # Header style
        header_fill = PatternFill("solid", fgColor="1F4E79")
        for cell in ws[1]:
            cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
            cell.fill      = header_fill
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        ws.row_dimensions[1].height = 30
        ws.freeze_panes = "A2"
    print(f"  Exported → {path}")

from datetime import datetime

export_date = datetime.now().strftime("%Y%m%d")
valid_path = os.path.join(VALID_DIR, f"valid_addresses_{export_date}.xlsx")
invalid_path = os.path.join(INVALID_DIR, f"invalid_addresses_{export_date}.xlsx")

write_excel(valid_out, valid_path, "Valid")
write_excel(invalid_out, invalid_path, "Invalid")

print("\nPipeline complete ✓")
print(f" Valid → {valid_path}")
print(f" Invalid → {invalid_path}")

  Exported → ../../data/output/valid\valid_addresses_20260420.xlsx
  Exported → ../../data/output/invalid\invalid_addresses_20260420.xlsx

Pipeline complete ✓
 Valid → ../../data/output/valid\valid_addresses_20260420.xlsx
 Invalid → ../../data/output/invalid\invalid_addresses_20260420.xlsx


# Task
Refine the address normalization and fuzzy matching pipeline to improve accuracy for ambiguous addresses, particularly for cases like 'Tagumpay Sindalan CSFP'. This involves loading and incorporating additional data from `dictionary.json` for enhanced location lookups, updating the `detect_city` function to generate city-province candidates when initial detection is ambiguous, and modifying the `match_address` function to prioritize barangay matching and iteratively disambiguate using city variants. Finally, re-run the updated pipeline and summarize the improvements, remaining challenges, and updated counts of valid and invalid addresses.

## Load and Prepare Blocking Dictionary

### Subtask:
Load the `dictionary.json` file and process its content to create lookup structures for enhanced location lookups and disambiguation.


# Task
Prepare reference data by adding cleaned versions of 'city_name', 'province_name', and 'barangay_name' as new columns ('city_clean', 'prov_clean', 'bgy_clean') to the `dim_raw` DataFrame.

## Prepare Reference Data with Cleaned Columns

### Subtask:
Add cleaned versions of 'city_name', 'province_name', and 'barangay_name' as new columns ('city_clean', 'prov_clean', 'bgy_clean') to the `dim_raw` DataFrame.


## Refine City Detection to Return Candidates

### Subtask:
Modify the `detect_city` function to leverage the `city_to_provinces` mapping, returning a list of `(canonical_city_name, canonical_province_name)` tuples for detected cities.
